# `TableParser`, `DelimTxtParser`, and `ExcelParser`

A `TableParser` reads tabular data from a file and normalises the column names to BDF standards.
Two concrete parsers are provided:
- **`DelimTxtParser`** – reads delimiter-separated text files (CSV, TSV, etc.)
- **`ExcelParser`** – reads a named sheet from an Excel workbook

Both expose the same interface: `matches_ext()`, `read_column_headings()`, `normalizer_score()`, and `read()`.

In [ ]:
from pathlib import Path
from bdf.table_parsers import DelimTxtParser, ExcelParser
from bdf.normalizers import TableNormalizer, Syn, DateTimeSyn

## `DelimTxtParser` — delimited text files

Construct a `DelimTxtParser` with a `TableNormalizer` that maps raw column names to BDF fields.
The Biologic BT-Lab file has tab-separated columns including `time/s`, `Ecell/V`, `I/mA`, and `cycle number`.
Unit-template syns like `Syn(root='time/{unit}')` match any column whose name fits the pattern with a recognised unit.

In [ ]:
biologic_file = Path("../tests/data/biologic/Sample_data_biologic_01_MB_CA1.txt")

biologic_parser = DelimTxtParser(
    normalizer=TableNormalizer(
        test_time_second=(Syn(root="time/{unit}"),),
        voltage_volt=(Syn(root="ecell/{unit}"),),
        current_ampere=(Syn(root="i/{unit}"),),
        cycle_count=(Syn(root="cycle number"),),
    )
)
biologic_parser

`matches_ext()` checks the file extension against the parser's accepted types

In [ ]:
print("matches .txt: ", biologic_parser.matches_ext(".txt"))
print("matches .xlsx:", biologic_parser.matches_ext(".xlsx"))

`read_column_headings()` sniffs only the header row — no data rows loaded


In [ ]:
biologic_parser.read_column_headings(biologic_file)

`normalizer_score()` counts how many columns the normalizer can resolve


In [ ]:
biologic_parser.normalizer_score(biologic_file)

`read()` returns a normalised polars LazyFrame with BDF-standard column names

In [ ]:
biologic_parser.read(biologic_file).collect()

## Graceful degradation — empty normalizer preserves source column names

When a `TableNormalizer()` with no synonyms is supplied, `read()` still works but
keeps the original column names from the file. This is useful for exploring an unfamiliar file.

In [ ]:
raw_parser = DelimTxtParser(normalizer=TableNormalizer())
raw_parser.read(biologic_file).collect()

## `ExcelParser` — Excel workbooks

`ExcelParser` reads a specific sheet from an `.xlsx` file.
Its extension handling is the inverse of `DelimTxtParser`: `.xlsx` returns `True`, `.csv` returns `False`.

In [ ]:
neware_file = Path("../tests/data/neware/sample_data_neware.xlsx")

neware_parser = ExcelParser(
    sheet_name="record",
    normalizer=TableNormalizer(
        test_time_second=(Syn(root="total time"),),
        voltage_volt=(Syn(root="voltage({unit})"),),
        current_ampere=(Syn(root="current({unit})"),),
        cycle_count=(Syn(root="cycle index"),),
    ),
)
neware_parser

In [ ]:
print("matches .xlsx:", neware_parser.matches_ext(".xlsx"))
print("matches .csv: ", neware_parser.matches_ext(".csv"))

In [ ]:
neware_parser.read_column_headings(neware_file)

In [ ]:
neware_parser.read(neware_file).collect()